# RAG Pipeline

In [139]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

## Reading PDF Files

In [140]:
#read all pdf 
def all_pdfs(pdf_dir):
    ##process all pdf in dir
    all_doc = []
    pdf_dir = Path(pdf_dir)

    #find all pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"found {len(pdf_files)} PDF files")


    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            
            
            # Add  addintional source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_doc.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")




        except Exception as e: 
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_doc)}")
    return all_doc


# Process all PDFs in the data directory
all_pdf_documents = all_pdfs("../data")

found 4 PDF files

Processing: Deafness and hearing loss.pdf
  ✓ Loaded 5 pages

Processing: Drowning.pdf
  ✓ Loaded 5 pages

Processing: Malaria.pdf
  ✓ Loaded 7 pages

Processing: Post COVID-19 condition (long COVID).pdf
  ✓ Loaded 6 pages

Total documents loaded: 23


In [141]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m147', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36', 'creationdate': '2026-04-16T07:48:59+00:00', 'source': '..\\data\\Deafness and hearing loss.pdf', 'file_path': '..\\data\\Deafness and hearing loss.pdf', 'total_pages': 5, 'format': 'PDF 1.4', 'title': 'Deafness and hearing loss', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-04-16T07:48:59+00:00', 'trapped': '', 'modDate': "D:20260416074859+00'00'", 'creationDate': "D:20260416074859+00'00'", 'page': 0, 'source_file': 'Deafness and hearing loss.pdf', 'file_type': 'pdf'}, page_content='Donate\n©\nDeafness and hearing loss\n3 March 2026\nالعربية中文\nFrançais\nРусский\nEspañol\nKey facts\nBy 2050, nearly 2.5\xa0billion people are projected to have some degree of hearing loss,\nand more than 700\xa0million will require hearing rehabilitation.\nAround 95.1 million children aged 5-19 years live with heari

In [142]:
all_pdf_documents[3].page_content[:]

'genetic counselling\nidentification and management of common ear conditions\noccupational hearing conservation\xa0programmes\xa0for noise and chemical exposure\nsafe listening strategies for the reduction of exposure to loud sounds in recreational\nsettings\nrational use of medicines to prevent ototoxic hearing loss.\xa0\xa0\nIdentification and management\nEarly identification\xa0of hearing loss and ear diseases is key to effective management.\nThis requires systematic screening for detection of hearing loss and related ear diseases in\nthose who are most at risk. This includes:\nnewborn babies and infants\npre-school and school-age children\npeople exposed to noise or chemicals at work\npeople receiving ototoxic medicines\nolder adults.\nHearing assessment and ear examination can\xa0be conducted in clinical and community\nsettings. Tools such as the hearWHO app, WHOears app and other technology-based\nsolutions make it possible to screen for ear diseases and hearing loss with limited

## Chunking

In [143]:
def split_documents(documents, chunk_size=70, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""],
    )

    split_docs = text_splitter.split_documents(documents)

    print(f"Original docs: {len(documents)}")
    print(f"Chunks created: {len(split_docs)}")

    return split_docs

In [144]:
chunks=split_documents(all_pdf_documents)
chunks

Original docs: 23
Chunks created: 1162


[Document(metadata={'producer': 'Skia/PDF m147', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36', 'creationdate': '2026-04-16T07:48:59+00:00', 'source': '..\\data\\Deafness and hearing loss.pdf', 'file_path': '..\\data\\Deafness and hearing loss.pdf', 'total_pages': 5, 'format': 'PDF 1.4', 'title': 'Deafness and hearing loss', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-04-16T07:48:59+00:00', 'trapped': '', 'modDate': "D:20260416074859+00'00'", 'creationDate': "D:20260416074859+00'00'", 'page': 0, 'source_file': 'Deafness and hearing loss.pdf', 'file_type': 'pdf'}, page_content='Donate\n©\nDeafness and hearing loss\n3 March 2026\nالعربية中文\nFrançais'),
 Document(metadata={'producer': 'Skia/PDF m147', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36', 'creationdate': '2026-04-16T07:48:59+00:00', 'source': '..\\data\\Dea

# embedding and vectorstroreDB

In [145]:
from langchain_huggingface import HuggingFaceEmbeddings

# Create embeddings model (wrapper around SentenceTransformer)
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5070.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [146]:
from langchain_community.vectorstores import FAISS


# Create FAISS vectorstore from Document objects
vectorstore = FAISS.from_documents(chunks, embeddings_model)

## Retriever Pipeline From VectorStore

In [147]:
# retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 7, "score_threshold": 0.5}
)


In [148]:
# query = "tell me about covid"
# docs = retriever.invoke(query)
# docs

In [149]:
# context = f" ".join([doc.page_content for doc in docs])
# print(context)

## Load LLM

In [150]:
from transformers import pipeline,GenerationConfig

# llm = pipeline(
#     "text-generation",
#     model="gpt2"
# )

In [151]:
gen_config = GenerationConfig(
    max_new_tokens=200,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    do_sample=True,
    repetition_penalty=2.0,  # Prevent repetition
    pad_token_id=50256,
    eos_token_id=50256,
)

In [152]:
# prompt = f""" You are a helpful AI assistant. Use the following context to answer the question accurately and concisely use 3rd person speach.
# Context: {context}

# Question: {query}

# Answer:"""
    
# result = llm(prompt, generation_config=gen_config)
# print(result[0]["generated_text"])

In [154]:
## single PipeLine Function


def rag_pipeline(query, context=None):
    """
    Complete RAG pipeline: retrieve documents and generate answer
    """
    if context is None:
        docs = retriever.invoke(query)
        context = " ".join([doc.page_content for doc in docs])
    
    prompt1 = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    
    result = llm(prompt, generation_config=gen_config)
    return result[0]["generated_text"]

# Usage
qu = "symptoms of malaria? "
answer = rag_pipeline(qu)
print(answer)

Use the following context to answer the question concisely.
Context:
Donate
©
Post COVID-19 condition
(long COVID)
26 February 2025 With increasing understanding of post COVID-19 condition, some More recent research shows the chances of developing post COVID-19 and funders to support research on post COVID-19 condition . Estimates largely come from people who suffered COVID-19 early

Question: tell me about covid

Answer: I am a researcher in one area (post COPD). This is where you can find many other useful information like your own experience or how things work with VCU's new data set for this issue which has been developed by us at UC Berkeley over several years here today! It will be interesting if we are able publish our findings before December 16th 2016 as part "The Next Stage" under The Enduring Future Project - A Global View." Our goal was not only that these studies should provide more reliable estimates but also help inform future projects because there needs always an impor